# 하천 쓰레기 분류 AI 프로그램 구현 · 4차시

**사용법**
1. 메뉴 **파일 → Drive에 사본 저장**을 눌러 내 노트북으로 만듭니다.
2. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 고릅니다. (없으면 CPU로도 됩니다)
3. **⓪ 실험 설정** 칸만 바꾸고, 위에서부터 셀을 차례로 실행(▶)합니다.
4. **1단계 · 공통 실험 (티처블 머신의 비밀 찾기)**
   - `USE_TRANSFER = False`(처음부터 배우는 CNN)로 한 번 실행합니다.
   - `USE_TRANSFER = True`(전이 학습)로 바꾸고 **⓪부터 다시 실행**합니다.
   - ⑦ 기록표의 두 줄을 비교합니다. 어느 쪽이 3차시 티처블 머신과 비슷한가요?
5. **2단계 · 개선 실험**
   - `USE_TRANSFER = False`로 되돌리고, 데이터 보강·데이터 증강 중 **하나만** `True`로 바꿔 다시 실행합니다.
   - 1단계 첫 줄(처음부터 CNN)과 비교합니다.
   - 도전: 전이 학습을 켠 채로 개선 전략을 더하면 티처블 머신을 넘을 수 있을까요?

⑦ 기록표는 런타임이 끊기면 사라집니다. 결과가 나오면 바로 개선 계획서에 옮겨 적으세요.

## ⓪ 실험 설정 — 이 칸만 바꿉니다!

In [ ]:
MY_NAME = "10301 김공주"      # 내 학번 이름

# ── 1단계 · 공통 실험: 티처블 머신의 비밀 ──
USE_TRANSFER = False   # 전이 학습: 사진을 잘 보는 모델(MobileNetV2)을 빌려 쓰기 (False 한 번, True 한 번)

# ── 2단계 · 개선 실험: 둘 중 하나만 True ──
ADD_DIRTY_PHOTOS = False   # 데이터 보강: 흙 묻은 사진 80장을 훈련 데이터에 추가
USE_AUGMENTATION = False   # 데이터 증강: 사진을 돌리고·확대하고·뒤집어서 여러 모습으로 보여 주기

# ── 학습 조건 (3차시 티처블 머신과 같은 값) ──
PHOTOS_PER_CLASS = 50       # 클래스당 사진 수 (10으로 바꾸면 3차시 실험 1과 같은 조건)
NUM_EPOCHS       = 50       # 학습 횟수 (5로 바꾸면 3차시 실험 3과 같은 조건)
LEARNING_RATE    = 0.0002   # AI가 학습할 때 나아가는 '보폭'

model_name = "전이 학습" if USE_TRANSFER else "처음부터 CNN"
strategy = [n for n, v in [("데이터 보강", ADD_DIRTY_PHOTOS), ("데이터 증강", USE_AUGMENTATION)] if v]
if len(strategy) > 1:
    print("⚠️ 개선 전략은 한 번에 하나만 켜야 무엇 때문에 결과가 달라졌는지 알 수 있어요!")
experiment = model_name + "".join(" + " + n for n in strategy)
print("이번 실험:", experiment, f"· 클래스당 사진 {PHOTOS_PER_CLASS}장 · 에포크 {NUM_EPOCHS}번")

## ① 필요한 라이브러리 불러오기

In [ ]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

print("TensorFlow Version:", tf.__version__)

## ② 데이터 준비하기
쓰레기 사진을 받아 와서 클래스마다 **PHOTOS_PER_CLASS장**을 고르고, **훈련 데이터 80%**와 **검증 데이터 20%**로 나눕니다.
(교재처럼 구글 드라이브를 쓰지 않고, 선생님이 올려 둔 사진을 자동으로 받습니다)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/moonyoungwoong/2026divdetreshfromriver/main/colab/river_trash_data.zip"
if not os.path.exists("river_trash"):
    urllib.request.urlretrieve(DATA_URL, "river_trash.zip")
    zipfile.ZipFile("river_trash.zip").extractall(".")

DATA_DIR  = "river_trash/trash_data"     # 클래스당 최대 50장 (깨끗한 사진)
DIRTY_DIR = "river_trash/dirty_extra"    # 데이터 보강용 흙 묻은 사진 (클래스당 20장)
TEST_DIR  = "river_trash/test"           # 3차시와 같은 공통 테스트 사진 8장

IMAGE_SIZE = (128, 128)   # 모든 사진을 가로 128, 세로 128 픽셀로 맞춤
BATCH_SIZE = 16           # 한 번에 AI에 넣는 사진 수
SEED = 123                # 무작위로 나눌 때 모두 같은 결과가 나오게 하는 값
AUTOTUNE = tf.data.AUTOTUNE
tf.keras.utils.set_random_seed(SEED)
CLASS_NAMES = sorted(os.listdir(DATA_DIR))
print("분류 클래스:", CLASS_NAMES)

def pick_photos(folder, n=None):
    # 클래스마다 사진을 n장 골라 훈련 80% · 검증 20%로 나눈다
    rng = np.random.default_rng(SEED)
    train, val = [], []
    for label, c in enumerate(CLASS_NAMES):
        files = sorted(f"{folder}/{c}/{f}" for f in os.listdir(f"{folder}/{c}") if f.lower().endswith(".jpg"))
        files = list(rng.permutation(files))[:n] if n else files
        k = max(1, round(len(files) * 0.2))
        val   += [(f, label) for f in files[:k]]
        train += [(f, label) for f in files[k:]]
    return train, val

def make_dataset(pairs, shuffle=False):
    paths, labels = zip(*pairs)
    ds = tf.data.Dataset.from_tensor_slices((list(paths), list(labels)))
    ds = ds.map(lambda p, l: (tf.image.resize(tf.io.decode_jpeg(tf.io.read_file(p), channels=3), IMAGE_SIZE), l),
                num_parallel_calls=AUTOTUNE).cache()
    if shuffle:
        ds = ds.shuffle(len(pairs), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_pairs, val_pairs = pick_photos(DATA_DIR, PHOTOS_PER_CLASS)
if ADD_DIRTY_PHOTOS:   # 흙 묻은 사진은 훈련 데이터에만 더한다 (검증 데이터는 그대로 두어 공정하게 비교)
    dirty_train, dirty_val = pick_photos(DIRTY_DIR)
    train_pairs += dirty_train + dirty_val
    print("➕ 흙 묻은 사진을 훈련 데이터에 추가했습니다.")

train_dataset = make_dataset(train_pairs, shuffle=True)
validation_dataset = make_dataset(val_pairs)
print(f"훈련 사진 {len(train_pairs)}장 · 검증 사진 {len(val_pairs)}장")

## ③ 모델 만들기
- 기본: 교재의 **CNN** (합성곱층으로 사진의 특징을 찾고, 완전 연결층으로 최종 분류)
- 전이 학습: 100만 장 넘는 사진으로 미리 학습된 **MobileNetV2**가 특징을 찾고, 마지막 판단만 새로 학습

In [ ]:
tf.keras.utils.set_random_seed(SEED)
input_shape = IMAGE_SIZE + (3,)
num_classes = len(CLASS_NAMES)

augment = [
    layers.RandomRotation(0.2, seed=SEED),
    layers.RandomZoom(0.2, seed=SEED),
    layers.RandomFlip("horizontal", seed=SEED),
] if USE_AUGMENTATION else []

if USE_TRANSFER:
    base = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    base.trainable = False   # 빌려 온 '눈'은 그대로 두고
    model = models.Sequential([
        layers.Input(shape=input_shape),
        *augment,
        layers.Rescaling(1./127.5, offset=-1),   # MobileNetV2가 원하는 -1 ~ 1 범위로 맞춤
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),   # 마지막 판단만 새로 학습
    ])
else:
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Rescaling(1./255),
        *augment,
        # 합성곱층: 이미지에서 특징 추출
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),          # 2D 데이터를 1D로 펼치기
        layers.Dropout(0.6),
        # 완전 연결층: 추출된 특징을 기반으로 최종 분류
        layers.Dense(128, activation="relu"),
        layers.Dense(num_classes, activation="softmax"),
    ])
model.summary()

## ④ 학습 방법 정하기 (컴파일)

In [ ]:
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=["accuracy"])

## ⑤ 학습시키고 그래프 보기

In [ ]:
print(f"--- AI 학습 시작 ({experiment} · Learning Rate: {LEARNING_RATE} · Epochs: {NUM_EPOCHS}) ---")
history = model.fit(train_dataset, validation_data=validation_dataset,
                    epochs=NUM_EPOCHS, verbose=2)   # 티처블 머신처럼 정해진 횟수만큼 끝까지 학습
print("--- AI 학습 완료 ---")

h = history.history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["accuracy"], label="train acc", color="#5170FF")
ax[0].plot(h["val_accuracy"], label="val acc", color="#FF3131")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epochs"); ax[0].set_ylim(0, 1.05); ax[0].legend()
ax[1].plot(h["loss"], label="train loss", color="#5170FF")
ax[1].plot(h["val_loss"], label="val loss", color="#FF3131")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epochs"); ax[1].legend()
plt.show()

val_acc = h["val_accuracy"][-1]
print(f"마지막({NUM_EPOCHS}번째) 에포크 · 훈련 정확도 {h['accuracy'][-1]:.0%}, 검증 정확도 {val_acc:.0%}")
print("두 선이 크게 벌어져 있다면 과적합! (3차시 그래프와 비교해 보세요)")

## ⑥ 공통 테스트 사진 8장으로 채점하기
3차시 티처블 머신 때와 **같은 8장**입니다. 짝수 번호는 흙 묻은 사진입니다.

In [ ]:
TEST_ANSWERS = {1: "plastic", 2: "plastic", 3: "can", 4: "can", 5: "glass", 6: "glass", 7: "other", 8: "other"}
rows, fig = [], plt.figure(figsize=(16, 4.5))
for no, answer in TEST_ANSWERS.items():
    img = tf.keras.utils.load_img(f"{TEST_DIR}/{no}.jpg", target_size=IMAGE_SIZE, interpolation="bilinear")
    prob = model.predict(np.expand_dims(tf.keras.utils.img_to_array(img), 0), verbose=0)[0]
    pred = CLASS_NAMES[int(np.argmax(prob))]
    ok = pred == answer
    rows.append({"사진": f"{no}" + (" (흙)" if no % 2 == 0 else ""), "실제값": answer, "AI 예측값": pred,
                 "확률": f"{prob.max():.0%}", "결과": "⭕" if ok else "❌"})
    a = fig.add_subplot(1, 8, no); a.imshow(img); a.axis("off")
    a.set_title(f"#{no} answer: {answer}\nAI: {pred} {prob.max():.0%}", color="#1D1D1F" if ok else "#FF3131", fontsize=9)
plt.show()

test_df = pd.DataFrame(rows)
score = int((test_df["결과"] == "⭕").sum())
clean = int(sum(r["결과"] == "⭕" for i, r in enumerate(rows) if i % 2 == 0))
dirty = score - clean
display(test_df)
print(f"테스트 점수: {score} / 8  (깨끗한 사진 {clean}/4, 흙 묻은 사진 {dirty}/4)")

## ⑦ 내 실험 기록표
실행할 때마다 **자동으로** 한 줄씩 쌓입니다. 1단계 두 줄(처음부터 CNN ↔ 전이 학습)과 2단계 줄을 비교해 개선 계획서와 발표에 씁니다.

In [ ]:
try:
    RESULTS
except NameError:
    RESULTS = []
RESULTS.append({
    "이름": MY_NAME,
    "실험": experiment,
    "사진(클래스당)": PHOTOS_PER_CLASS,
    "에포크": NUM_EPOCHS,
    "학습률": LEARNING_RATE,
    "검증 정확도": f"{val_acc:.0%}",
    "테스트 점수(/8)": score,
    "흙 묻은 사진(/4)": dirty,
})
display(pd.DataFrame(RESULTS))